In [1]:
import os
import glob
import json
import numpy as np
from skimage import io

def evaluate_single_image(eval_image_path, detected_dict=None):
    nom_fichier = os.path.basename(eval_image_path)
    image_number = int(nom_fichier.split('_')[1].split('.')[0])
    
    # 1. Résolution des chemins
    result_dir = os.path.dirname(eval_image_path)
    sim_dir = os.path.dirname(result_dir)
    seq_dir = os.path.dirname(sim_dir)
    
    gt_image_path = os.path.join(seq_dir, "low dyn", nom_fichier)
    def_image_path = os.path.join(sim_dir, nom_fichier) # NOUVEAU : Chemin vers l'image défectueuse
    
    # 2. Chargement du JSON
    fichiers_json = glob.glob(os.path.join(sim_dir, "*.json"))
    if not fichiers_json:
        raise FileNotFoundError(f"Aucun fichier JSON trouvé dans le dossier : {sim_dir}")
    json_path = fichiers_json[0]
    
    with open(json_path, "r", encoding="utf-8") as file:
        json_data = json.load(file)
        
    # 3. Extraction des défauts du JSON
    true_def_dict = {}
    types_presents = set()
    
    for item in json_data:
        value = item.get('signal', {}).get(str(image_number))
        if value is not None:
            def_type = item.get('name', 'unknown')
            if isinstance(def_type, list) and len(def_type) > 0:
                def_type = def_type[0]
            types_presents.add(def_type)
            
            for x_coord in item.get('x_coord', []):
                true_def_dict[x_coord] = {
                    'start': item.get('y_start', []), 
                    'stop': item.get('y_stop', []),
                    'type': def_type
                }
                
    # 4. Chargement des 3 images (Vérité Terrain, Défectueuse, Corrigée)
    gt = io.imread(gt_image_path).astype(np.float32, copy=False)
    def_img = io.imread(def_image_path).astype(np.float32, copy=False) # NOUVEAU
    res = io.imread(eval_image_path).astype(np.float32, copy=False)
    
    # 5. Création des masques de Vérité (Ground Truth)
    true_mask_global = np.zeros(gt.shape, dtype=bool)
    masks_by_type = {t: np.zeros(gt.shape, dtype=bool) for t in types_presents}
    
    for col_str, info in true_def_dict.items():
        col_idx = int(col_str)
        def_type = info.get('type', 'unknown')
        for start, stop in zip(info.get('start', []), info.get('stop', [])):
            true_mask_global[start:stop, col_idx] = True
            masks_by_type[def_type][start:stop, col_idx] = True
            
    # 6. Création du masque de Détection (MODIFIÉ SELON LE NOUVEAU METRICS.PY)
    if detected_dict is None:
        # On considère qu'un pixel est détecté si sa valeur a été modifiée 
        # par rapport à l'image défectueuse d'origine (et non par rapport à la GT)
        detected_mask = (res != def_img) 
    else:
        detected_mask = np.zeros(gt.shape, dtype=bool)
        for col_str, info in detected_dict.items():
            col_idx = int(col_str)
            for start, stop in zip(info.get('start', []), info.get('stop', [])):
                detected_mask[start:stop, col_idx] = True
                
    # 7. Calcul des erreurs RMSE (Toujours calculé par rapport à la GT)
    residu_sq = (res - gt) ** 2
    
    # 8. Calcul des métriques Globales
    tp_global = np.logical_and(detected_mask, true_mask_global).sum()
    fp_global = np.logical_and(detected_mask, ~true_mask_global).sum()
    fn_global = np.logical_and(~detected_mask, true_mask_global).sum()
    
    prec_global = tp_global / (tp_global + fp_global) if (tp_global + fp_global) > 0 else 0.0
    rec_global  = tp_global / (tp_global + fn_global) if (tp_global + fn_global) > 0 else 0.0
    f1_global   = 2 * prec_global * rec_global / (prec_global + rec_global) if (prec_global + rec_global) > 0 else 0.0
    
    sq_def_global = residu_sq[true_mask_global].sum()
    cnt_def_global = true_mask_global.sum()
    sq_ok_global = residu_sq[~true_mask_global].sum()
    cnt_ok_global = (~true_mask_global).sum()
    
    rmse_def_global = np.sqrt(sq_def_global / cnt_def_global) if cnt_def_global > 0 else 0.0
    rmse_ok_global  = np.sqrt(sq_ok_global / cnt_ok_global) if cnt_ok_global > 0 else 0.0
    
    rmse_def_norm = max(0.0, 1.0 - (rmse_def_global / 40.0))
    rmse_ok_norm  = max(0.0, 1.0 - (rmse_ok_global / 40.0))
    overall = (0.34 * f1_global + 0.33 * rmse_def_norm + 0.33 * rmse_ok_norm)
    
    results = {
        "Global": {
            "TP": int(tp_global), "FP": int(fp_global), "FN": int(fn_global),
            "Precision": prec_global, "Recall": rec_global, "F1_Score": f1_global,
            "RMSE_def": rmse_def_global, "RMSE_ok": rmse_ok_global,
            "Overall_Score": overall
        },
        "By_Type": {}
    }
    
    # 9. Calcul des métriques par Type
    for def_type, type_mask in masks_by_type.items():
        tp_t = np.logical_and(detected_mask, type_mask).sum()
        fn_t = np.logical_and(~detected_mask, type_mask).sum()
        rec_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0.0
        
        sq_def_t = residu_sq[type_mask].sum()
        cnt_def_t = type_mask.sum()
        rmse_def_t = np.sqrt(sq_def_t / cnt_def_t) if cnt_def_t > 0 else 0.0
        
        results["By_Type"][def_type] = {
            "TP": int(tp_t),
            "FN": int(fn_t),
            "Recall": rec_t,
            "RMSE_def": rmse_def_t
        }
        
    # 10. Affichage
    print("=== RÉSULTATS GLOBAUX ===")
    print(f"TP: {results['Global']['TP']} | FP: {results['Global']['FP']} | FN: {results['Global']['FN']}")
    print(f"Precision: {results['Global']['Precision']:.4f} | Recall: {results['Global']['Recall']:.4f} | F1: {results['Global']['F1_Score']:.4f}")
    print(f"RMSE_def: {results['Global']['RMSE_def']:.2f} | RMSE_ok: {results['Global']['RMSE_ok']:.2f}")
    print(f"Score Final (Overall): {results['Global']['Overall_Score']:.4f}")
    print("\n=== RÉSULTATS PAR TYPE DE DÉFAUT ===")
    for def_type, metrics in results["By_Type"].items():
        print(f"[{def_type.upper()}]")
        print(f"  TP: {metrics['TP']} | FN: {metrics['FN']}")
        print(f"  Recall (Rappel): {metrics['Recall']:.4f} | RMSE_def: {metrics['RMSE_def']:.2f}")
    print("====================================")
    
    return results

In [7]:
scrores = evaluate_single_image(r"C:\Users\eliot\Desktop\Obsidian Vault\03. Cours\S2\Data_Challenge\train\VGA\sequence_1\low dyn with columns 2\result\frame_0000.png")

=== RÉSULTATS GLOBAUX ===
TP: 0 | FP: 0 | FN: 1029
Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
RMSE_def: 38.98 | RMSE_ok: 0.00
Score Final (Overall): 0.3384

=== RÉSULTATS PAR TYPE DE DÉFAUT ===
[BLINKING]
  TP: 0 | FN: 457
  Recall (Rappel): 0.0000 | RMSE_def: 47.45
[NOISY]
  TP: 0 | FN: 572
  Recall (Rappel): 0.0000 | RMSE_def: 30.58
